# Natural Language Processing Lab - Assignment 01
## Task 3: Spelling Error Detection & Correction using Edit Distance

**Objective**:
1. Implement a minimum edit distance spelling corrector using the NLTK English words corpus.
2. Evaluate correction accuracy on 10 intentionally misspelled terms.
3. Apply sentence-level correction across 5 test sentences.

In [ ]:
import nltk
from nltk.corpus import words
import pandas as pd

nltk.download('words', quiet=True)

# Build English vocabulary index
english_vocab_set = set(w.lower() for w in words.words())
print(f"Loaded English lexicon with {len(english_vocab_set):,} distinct entries.")

### 1. Minimum Edit Distance Correction Function

In [ ]:
def correct_spelling(term: str, vocabulary: set[str], length_tolerance: int = 2) -> str:
    """
    Finds the closest English word to the query string based on Levenshtein edit distance.
    """
    target = term.lower()
    if target in vocabulary:
        return target
        
    # Candidate filtering by character length bounds
    candidates = [
        w for w in vocabulary 
        if abs(len(w) - len(target)) <= length_tolerance and (w[0] == target[0] if len(target) > 3 else True)
    ]
    
    if not candidates:
        candidates = [w for w in vocabulary if abs(len(w) - len(target)) <= length_tolerance]
        
    # Sort candidate words by edit distance
    candidates.sort(key=lambda word: nltk.edit_distance(target, word))
    return candidates[0] if candidates else target

### 2. Testing on 10 Benchmark Misspelled Words

In [ ]:
test_misspelled_words = [
    'recieve',
    'becaus',
    'teh',
    'studys',
    'langauge',
    'computr',
    'progrmming',
    'lernning',
    'importent',
    'enviroment'
]

spellcheck_records = []
for misspelled in test_misspelled_words:
    suggested = correct_spelling(misspelled, english_vocab_set)
    dist = nltk.edit_distance(misspelled.lower(), suggested)
    spellcheck_records.append({
        "Misspelled Input": misspelled,
        "Suggested Correction": suggested,
        "Edit Distance": dist
    })

spellcheck_df = pd.DataFrame(spellcheck_records)
display(spellcheck_df)

### 3. Sentence-Level Spell Correction Pipeline

In [ ]:
sample_faulty_sentences = [
    "I recieve my computr today.",
    "She is lerning Python progrmming.",
    "This is very importent for my studys.",
    "The langauge model needs a good enviroment.",
    "I stayed home becaus the weather was bad."
]

def correct_sentence(sentence: str) -> str:
    words_in_sent = sentence.split()
    fixed_tokens = []
    for token in words_in_sent:
        # Preserve trailing punctuation if present
        trailing_punct = ""
        clean_token = token
        if token and token[-1] in ".!?,":
            trailing_punct = token[-1]
            clean_token = token[:-1]
            
        corrected = correct_spelling(clean_token, english_vocab_set)
        
        # Preserve original capitalization
        if clean_token.istitle():
            corrected = corrected.capitalize()
            
        fixed_tokens.append(corrected + trailing_punct)
    return " ".join(fixed_tokens)

for idx, sent in enumerate(sample_faulty_sentences, 1):
    print(f"Sentence {idx}:")
    print(f"  Original  : {sent}")
    print(f"  Corrected : {correct_sentence(sent)}\n")